# Model Serving and Production Optimization

**Module 15: LLM Systems Design**  
**Objective**: Master production LLM deployment techniques

Production LLM Systems:
- Model Serving Architectures
- Quantization (INT8, INT4, GPTQ, AWQ)
- Flash Attention
- Distributed Inference
- Tensor Parallelism
- Pipeline Parallelism
- Cost Optimization

## What You'll Learn
1. Model serving architectures
2. Quantization for memory reduction
3. Flash Attention for speed
4. Distributed inference strategies
5. Parallelism techniques
6. Production cost optimization

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple, Optional, Dict
from dataclasses import dataclass
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}\n")

torch.manual_seed(42)
np.random.seed(42)

## 1. Model Serving Architecture

**Production LLM Serving Stack**:

```
┌─────────────────────────────────────┐
│      Application Layer              │
│  (FastAPI, gRPC, WebSocket)         │
└─────────────────────────────────────┘
           ↓                           
┌─────────────────────────────────────┐
│     Request Queue & Scheduler       │
│  (Priority, Rate Limiting, Routing) │
└─────────────────────────────────────┘
           ↓                           
┌─────────────────────────────────────┐
│     Inference Engine                │
│  (vLLM, TGI, TensorRT-LLM)         │
└─────────────────────────────────────┘
           ↓                           
┌─────────────────────────────────────┐
│     Model & Hardware                │
│  (GPU: A100, H100, TPU)            │
└─────────────────────────────────────┘
```

**Key Components**:
1. **Load Balancer**: Distribute requests across replicas
2. **Request Queue**: Buffer and prioritize requests
3. **Scheduler**: Continuous batching, resource allocation
4. **Inference Engine**: Optimized model execution
5. **Monitoring**: Latency, throughput, errors, costs

In [ ]:
def visualize_serving_architecture():
    """Visualize production serving architecture."""
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    
    # 1. Single GPU Serving
    ax1 = axes[0]
    ax1.set_title('Single GPU Serving', fontsize=14, weight='bold')
    ax1.axis('off')
    ax1.set_xlim(0, 1)
    ax1.set_ylim(0, 1)
    
    # Clients
    for i in range(3):
        x = 0.15 + i * 0.25
        circle = plt.Circle((x, 0.85), 0.04, color='lightblue',
                           ec='black', linewidth=2)
        ax1.add_patch(circle)
        ax1.text(x, 0.85, f'C{i+1}', ha='center', va='center',
                fontsize=10, weight='bold')
        
        # Arrows to load balancer
        ax1.annotate('', xy=(0.5, 0.72), xytext=(x, 0.81),
                    arrowprops=dict(arrowstyle='->', lw=2, color='blue'))
    
    # Load Balancer
    rect = plt.Rectangle((0.35, 0.65), 0.3, 0.08,
                         facecolor='lightgreen', edgecolor='black', linewidth=2)
    ax1.add_patch(rect)
    ax1.text(0.5, 0.69, 'Load Balancer', ha='center', va='center',
            fontsize=11, weight='bold')
    
    ax1.annotate('', xy=(0.5, 0.58), xytext=(0.5, 0.65),
                arrowprops=dict(arrowstyle='->', lw=3, color='black'))
    
    # Request Queue
    rect = plt.Rectangle((0.35, 0.5), 0.3, 0.08,
                         facecolor='lightyellow', edgecolor='black', linewidth=2)
    ax1.add_patch(rect)
    ax1.text(0.5, 0.54, 'Request Queue', ha='center', va='center',
            fontsize=11, weight='bold')
    
    ax1.annotate('', xy=(0.5, 0.43), xytext=(0.5, 0.5),
                arrowprops=dict(arrowstyle='->', lw=3, color='black'))
    
    # Inference Engine
    rect = plt.Rectangle((0.35, 0.3), 0.3, 0.13,
                         facecolor='lightcoral', edgecolor='black', linewidth=3)
    ax1.add_patch(rect)
    ax1.text(0.5, 0.39, 'Inference Engine\n(vLLM)', ha='center', va='center',
            fontsize=11, weight='bold')
    
    ax1.annotate('', xy=(0.5, 0.23), xytext=(0.5, 0.3),
                arrowprops=dict(arrowstyle='->', lw=3, color='black'))
    
    # GPU
    rect = plt.Rectangle((0.35, 0.1), 0.3, 0.13,
                         facecolor='gold', edgecolor='black', linewidth=3)
    ax1.add_patch(rect)
    ax1.text(0.5, 0.19, 'GPU\n(A100 80GB)', ha='center', va='center',
            fontsize=11, weight='bold')
    
    # Metrics
    ax1.text(0.5, 0.02, 'Throughput: ~1K tokens/sec\nLatency: 50-100ms',
            ha='center', fontsize=9, style='italic',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    # 2. Multi-GPU Distributed Serving
    ax2 = axes[1]
    ax2.set_title('Multi-GPU Distributed Serving', fontsize=14, weight='bold')
    ax2.axis('off')
    ax2.set_xlim(0, 1)
    ax2.set_ylim(0, 1)
    
    # Clients
    for i in range(5):
        x = 0.1 + i * 0.18
        circle = plt.Circle((x, 0.85), 0.03, color='lightblue',
                           ec='black', linewidth=2)
        ax2.add_patch(circle)
        ax2.text(x, 0.85, f'C{i+1}', ha='center', va='center',
                fontsize=9, weight='bold')
        
        # Arrows
        target_x = 0.25 if i < 2 else (0.5 if i < 4 else 0.75)
        ax2.annotate('', xy=(target_x, 0.72), xytext=(x, 0.82),
                    arrowprops=dict(arrowstyle='->', lw=1.5, color='blue'))
    
    # Multiple replicas
    replicas = [
        (0.15, 'vLLM\nReplica 1'),
        (0.5, 'vLLM\nReplica 2'),
        (0.85, 'vLLM\nReplica 3')
    ]
    
    for x, label in replicas:
        # Inference engine
        rect = plt.Rectangle((x-0.1, 0.55), 0.2, 0.17,
                            facecolor='lightcoral', edgecolor='black', linewidth=2)
        ax2.add_patch(rect)
        ax2.text(x, 0.635, label, ha='center', va='center',
                fontsize=9, weight='bold')
        
        ax2.annotate('', xy=(x, 0.48), xytext=(x, 0.55),
                    arrowprops=dict(arrowstyle='->', lw=2, color='black'))
        
        # GPUs
        for i in range(2):
            gpu_x = x - 0.06 + i * 0.12
            rect = plt.Rectangle((gpu_x-0.04, 0.35), 0.08, 0.13,
                                facecolor='gold', edgecolor='black', linewidth=2)
            ax2.add_patch(rect)
            ax2.text(gpu_x, 0.415, f'GPU{i+1}', ha='center', va='center',
                    fontsize=8, weight='bold')
    
    # Distributed communication
    ax2.plot([0.15, 0.5, 0.85], [0.3, 0.3, 0.3], 'g--', linewidth=2, alpha=0.5)
    ax2.text(0.5, 0.25, 'NCCL/NVLink (GPU-GPU communication)',
            ha='center', fontsize=9, color='green', style='italic')
    
    # Metrics
    ax2.text(0.5, 0.12, 'Throughput: ~8K tokens/sec (8x)\nLatency: 60-120ms',
            ha='center', fontsize=9, style='italic',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    ax2.text(0.5, 0.02, '✓ Horizontal scaling  ✓ High availability  ✓ Load balancing',
            ha='center', fontsize=9, weight='bold', color='green')
    
    plt.tight_layout()
    plt.show()
    
    print("Model Serving Architecture:")
    print("="*70)
    
    print("\nSERVING STRATEGIES:")
    
    print("\n1. Single GPU:")
    print("   • Simplest setup")
    print("   • Good for: Small models (<70B), low traffic")
    print("   • Cost: ~$2-4/hour (A100)")
    print("   • Throughput: 1-2K tokens/sec")
    
    print("\n2. Multi-GPU (Replica):
    print("   • Horizontal scaling")
    print("   • Good for: High traffic, fault tolerance")
    print("   • Cost: N × single GPU")
    print("   • Throughput: N × single GPU")
    
    print("\n3. Distributed (Tensor Parallel):")
    print("   • Model sharded across GPUs")
    print("   • Good for: Very large models (>70B)")
    print("   • Cost: N × GPU (but required for large models)")
    print("   • Latency: Similar to single GPU")
    
    print("\nBEST PRACTICES:")
    print("  ✓ Use continuous batching (vLLM, TGI)")
    print("  ✓ Monitor latency P50, P95, P99")
    print("  ✓ Set timeouts and rate limits")
    print("  ✓ Implement caching for common queries")
    print("  ✓ Use autoscaling for variable traffic")
    print("  ✓ Deploy multiple regions for latency")

visualize_serving_architecture()

## 2. Quantization

**Quantization** reduces model memory and computation by using lower precision

**Precision Formats**:
- **FP32**: 32-bit float (default training)
- **FP16/BF16**: 16-bit float (common inference)
- **INT8**: 8-bit integer (2x memory reduction)
- **INT4**: 4-bit integer (4x memory reduction)

**Quantization Methods**:

1. **Post-Training Quantization (PTQ)**:
   - Quantize trained model
   - Fast, no retraining
   - May have accuracy loss

2. **Quantization-Aware Training (QAT)**:
   - Train with quantization in loop
   - Better accuracy
   - More expensive

**Popular Methods**:
- **GPTQ**: Optimal brain compression
- **AWQ**: Activation-aware weight quantization
- **GGUF**: llama.cpp format
- **bitsandbytes**: 8-bit & 4-bit (QLoRA)

In [ ]:
def quantize_tensor(tensor: torch.Tensor, bits: int = 8) -> Tuple[torch.Tensor, float, float]:
    """
    Simple symmetric quantization.
    
    Q = round(clip(X / scale, -2^(bits-1), 2^(bits-1)-1))
    
    Args:
        tensor: FP32 tensor to quantize
        bits: Number of bits (4 or 8)
        
    Returns:
        quantized: Quantized tensor
        scale: Quantization scale
        zero_point: Zero point (for symmetric, always 0)
    """
    # Compute scale
    abs_max = tensor.abs().max().item()
    qmax = 2 ** (bits - 1) - 1
    scale = abs_max / qmax if abs_max > 0 else 1.0
    
    # Quantize
    quantized = torch.round(tensor / scale).clamp(-qmax - 1, qmax)
    
    if bits == 8:
        quantized = quantized.to(torch.int8)
    else:
        quantized = quantized.to(torch.int32)  # PyTorch doesn't have int4
    
    return quantized, scale, 0.0

def dequantize_tensor(quantized: torch.Tensor, scale: float) -> torch.Tensor:
    """Dequantize back to FP32."""
    return quantized.float() * scale

# Example quantization
print("\nQuantization Example:")
print("="*70)

# Create example weight matrix
np.random.seed(42)
weights_fp32 = torch.randn(4, 4)

print("\nOriginal FP32 weights:")
print(weights_fp32)
print(f"Memory: {weights_fp32.numel() * 4} bytes ({weights_fp32.numel() * 4 / 1024:.2f} KB)")

# INT8 quantization
weights_int8, scale_8, _ = quantize_tensor(weights_fp32, bits=8)
weights_dequant_8 = dequantize_tensor(weights_int8, scale_8)

print("\nINT8 quantized:")
print(weights_int8)
print(f"Scale: {scale_8:.6f}")
print(f"Memory: {weights_int8.numel() * 1} bytes ({weights_int8.numel() * 1 / 1024:.2f} KB)")
print(f"Memory reduction: {4.0 / 1.0:.1f}x")

print("\nDequantized (INT8 → FP32):")
print(weights_dequant_8)

# Quantization error
error_8 = (weights_fp32 - weights_dequant_8).abs().max().item()
print(f"\nMax error (INT8): {error_8:.6f}")

# INT4 quantization
weights_int4, scale_4, _ = quantize_tensor(weights_fp32, bits=4)
weights_dequant_4 = dequantize_tensor(weights_int4, scale_4)

print("\nINT4 quantized values (stored as int32):")
print(weights_int4)
print(f"Scale: {scale_4:.6f}")
print(f"Actual memory: {weights_int4.numel() * 0.5:.0f} bytes (packed)")
print(f"Memory reduction: {4.0 / 0.5:.1f}x")

error_4 = (weights_fp32 - weights_dequant_4).abs().max().item()
print(f"\nMax error (INT4): {error_4:.6f}")

# Visualize quantization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Original distribution
ax1 = axes[0, 0]
ax1.set_title('Original FP32 Weights', fontsize=12, weight='bold')
weights_flat = weights_fp32.flatten().numpy()
ax1.hist(weights_flat, bins=20, color='blue', alpha=0.7, edgecolor='black')
ax1.set_xlabel('Weight Value')
ax1.set_ylabel('Frequency')
ax1.grid(True, alpha=0.3)

# INT8 quantized
ax2 = axes[0, 1]
ax2.set_title('INT8 Quantized', fontsize=12, weight='bold')
weights_8_flat = weights_dequant_8.flatten().numpy()
ax2.hist(weights_8_flat, bins=20, color='green', alpha=0.7, edgecolor='black')
ax2.set_xlabel('Weight Value')
ax2.set_ylabel('Frequency')
ax2.grid(True, alpha=0.3)

# INT4 quantized
ax3 = axes[1, 0]
ax3.set_title('INT4 Quantized', fontsize=12, weight='bold')
weights_4_flat = weights_dequant_4.flatten().numpy()
ax3.hist(weights_4_flat, bins=20, color='red', alpha=0.7, edgecolor='black')
ax3.set_xlabel('Weight Value')
ax3.set_ylabel('Frequency')
ax3.grid(True, alpha=0.3)

# Comparison
ax4 = axes[1, 1]
ax4.set_title('Memory & Accuracy Tradeoff', fontsize=12, weight='bold')

methods = ['FP32', 'FP16', 'INT8', 'INT4']
memory = [4, 2, 1, 0.5]  # bytes per parameter
accuracy_loss = [0, 0.1, 0.5, 2.0]  # relative perplexity increase (%)

x = np.arange(len(methods))
width = 0.35

ax4_twin = ax4.twinx()

bars1 = ax4.bar(x - width/2, memory, width, label='Memory (bytes)',
               color='blue', alpha=0.7, edgecolor='black')
bars2 = ax4_twin.bar(x + width/2, accuracy_loss, width, label='Accuracy Loss (%)',
                     color='red', alpha=0.7, edgecolor='black')

ax4.set_xlabel('Quantization Method')
ax4.set_ylabel('Memory (bytes/param)', color='blue')
ax4_twin.set_ylabel('Accuracy Loss (%)', color='red')
ax4.set_xticks(x)
ax4.set_xticklabels(methods)
ax4.tick_params(axis='y', labelcolor='blue')
ax4_twin.tick_params(axis='y', labelcolor='red')
ax4.grid(True, alpha=0.3)

# Add value labels
for i, (m, a) in enumerate(zip(memory, accuracy_loss)):
    ax4.text(i - width/2, m + 0.1, f'{m}', ha='center', fontsize=9, weight='bold')
    ax4_twin.text(i + width/2, a + 0.1, f'{a}', ha='center', fontsize=9, weight='bold')

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("QUANTIZATION METHODS:")

print("\n1. GPTQ (Optimal Brain Compression):")
print("   • One-shot quantization")
print("   • Minimizes per-layer error")
print("   • Popular for LLaMA, Mistral")
print("   • 4-bit: ~4x memory reduction, <1% accuracy loss")

print("\n2. AWQ (Activation-Aware Weight Quantization):")
print("   • Protects salient weights")
print("   • Uses activation statistics")
print("   • Better accuracy than GPTQ")
print("   • Slightly slower inference")

print("\n3. bitsandbytes (QLoRA):")
print("   • 4-bit NormalFloat (NF4)")
print("   • Enables training on consumer GPUs")
print("   • Double quantization")
print("   • Used for LoRA fine-tuning")

print("\n4. GGUF (llama.cpp):")
print("   • CPU/edge inference")
print("   • Multiple quantization levels (Q2-Q8)")
print("   • Good for MacBooks, mobile")
print("   • Slower than GPU but portable")

print("\nMODEL SIZE EXAMPLES (LLaMA-70B):")
print("  • FP32: ~280GB")
print("  • FP16: ~140GB")
print("  • INT8: ~70GB")
print("  • INT4: ~35GB  ← Fits on single A100!")

## 3. Flash Attention

**Flash Attention** is an optimized attention algorithm

**Standard Attention** (O(N²) memory):
```python
S = Q @ K.T / sqrt(d)     # (N, N) - large!
P = softmax(S)
O = P @ V
```

**Problem**: Materializes large (N × N) attention matrix
- For N=2048, d=128: S is 16MB per head
- 32 heads × 32 layers = 16GB just for attention!

**Flash Attention** innovations:
1. **Tiling**: Compute in blocks that fit in SRAM
2. **Recomputation**: Don't store intermediate S, P matrices
3. **Fused kernels**: Combine operations in single GPU kernel

**Benefits**:
- 2-4x faster than standard attention
- Sub-linear memory growth
- Enables longer sequences
- Mathematically equivalent (exact same output)

In [ ]:
def standard_attention(Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor) -> torch.Tensor:
    """
    Standard attention (high memory).
    
    Args:
        Q, K, V: (batch, seq_len, d_model)
        
    Returns:
        output: (batch, seq_len, d_model)
    """
    d_k = Q.size(-1)
    
    # (batch, seq_len, seq_len) - O(N²) memory!
    scores = Q @ K.transpose(-2, -1) / (d_k ** 0.5)
    attn = F.softmax(scores, dim=-1)
    output = attn @ V
    
    return output

def flash_attention_simulation(Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor,
                               block_size: int = 64) -> torch.Tensor:
    """
    Simplified Flash Attention (for illustration).
    
    Real Flash Attention is implemented in CUDA.
    This is a Python simulation for educational purposes.
    """
    batch, seq_len, d_model = Q.shape
    d_k = d_model
    
    # Output accumulator
    output = torch.zeros_like(Q)
    
    # Process in blocks
    num_blocks = (seq_len + block_size - 1) // block_size
    
    for i in range(num_blocks):
        start_i = i * block_size
        end_i = min((i + 1) * block_size, seq_len)
        
        Q_block = Q[:, start_i:end_i, :]
        
        # Compute attention for this block
        for j in range(num_blocks):
            start_j = j * block_size
            end_j = min((j + 1) * block_size, seq_len)
            
            K_block = K[:, start_j:end_j, :]
            V_block = V[:, start_j:end_j, :]
            
            # Compute attention scores for this block pair
            scores_block = Q_block @ K_block.transpose(-2, -1) / (d_k ** 0.5)
            attn_block = F.softmax(scores_block, dim=-1)
            output_block = attn_block @ V_block
            
            output[:, start_i:end_i, :] += output_block
    
    return output

# Compare standard vs flash attention
print("\nFlash Attention Comparison:")
print("="*70)

batch_size = 2
seq_len = 512
d_model = 128

Q = torch.randn(batch_size, seq_len, d_model)
K = torch.randn(batch_size, seq_len, d_model)
V = torch.randn(batch_size, seq_len, d_model)

print(f"\nInput: batch={batch_size}, seq_len={seq_len}, d_model={d_model}")

# Standard attention memory
attention_matrix_size = batch_size * seq_len * seq_len * 4 / 1024 / 1024  # MB
print(f"\nStandard Attention:")
print(f"  Attention matrix: ({batch_size}, {seq_len}, {seq_len})")
print(f"  Memory: {attention_matrix_size:.2f} MB")
print(f"  Complexity: O(N²) memory")

# Flash attention memory (much smaller)
block_size = 64
flash_memory = batch_size * block_size * block_size * 4 / 1024 / 1024  # MB
print(f"\nFlash Attention:")
print(f"  Block size: {block_size}")
print(f"  Block memory: ({batch_size}, {block_size}, {block_size})")
print(f"  Memory: {flash_memory:.2f} MB")
print(f"  Memory reduction: {attention_matrix_size / flash_memory:.1f}x")
print(f"  Complexity: O(N) memory (streamed)")

# Timing comparison (CPU, for illustration)
import time

start = time.time()
out_standard = standard_attention(Q, K, V)
time_standard = time.time() - start

print(f"\nTiming (CPU):")
print(f"  Standard attention: {time_standard*1000:.2f} ms")
print(f"  (Flash attention is GPU-only, 2-4x faster on GPU)")

# Visualize memory usage
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Memory comparison
ax1 = axes[0]
ax1.set_title('Memory Usage: Standard vs Flash Attention', fontsize=13, weight='bold')

seq_lengths = [512, 1024, 2048, 4096, 8192]
memory_standard = [(s * s * 4 / 1024 / 1024) for s in seq_lengths]  # MB
memory_flash = [(64 * 64 * 4 / 1024 / 1024) for s in seq_lengths]  # MB (constant)

ax1.plot(seq_lengths, memory_standard, 'r-o', linewidth=2, markersize=8,
        label='Standard (O(N²))')
ax1.plot(seq_lengths, memory_flash, 'g-s', linewidth=2, markersize=8,
        label='Flash (O(1))')

ax1.set_xlabel('Sequence Length', fontsize=11)
ax1.set_ylabel('Memory (MB per head)', fontsize=11)
ax1.set_yscale('log')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Speedup
ax2 = axes[1]
ax2.set_title('Flash Attention Benefits', fontsize=13, weight='bold')

metrics = ['Memory\nSaving', 'Speed\nImprovement', 'Max Seq\nLength']
values = [8, 3, 4]  # 8x memory, 3x speed, 4x longer sequences
colors = ['blue', 'green', 'orange']

bars = ax2.bar(metrics, values, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
ax2.set_ylabel('Improvement (x)', fontsize=11)
ax2.set_ylim(0, max(values) * 1.2)

for bar, val in zip(bars, values):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{val}x',
            ha='center', va='bottom', fontsize=13, weight='bold')

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("FLASH ATTENTION FEATURES:")

print("\n1. Memory Efficiency:")
print("   • Standard: O(N²) memory for attention matrix")
print("   • Flash: O(N) memory through tiling")
print("   • Enables 4-16x longer sequences")

print("\n2. Speed:")
print("   • 2-4x faster than standard attention")
print("   • Fused CUDA kernels")
print("   • Better GPU utilization")

print("\n3. Numerical Stability:")
print("   • Online softmax for stability")
print("   • Exact same output as standard attention")
print("   • No approximation")

print("\n4. Adoption:")
print("   • vLLM: Built-in Flash Attention 2")
print("   • PyTorch 2.0+: torch.nn.functional.scaled_dot_product_attention")
print("   • Transformers: flash_attention_2=True")
print("   • TensorRT-LLM: Optimized Flash Attention")

## 4. Distributed Inference

**For large models (>70B parameters), single GPU isn't enough**

**Parallelism Strategies**:

### Tensor Parallelism (TP)
- Split model layers across GPUs
- Each GPU computes part of each layer
- Low latency (GPUs work together)
- Example: 70B model on 4×A100

### Pipeline Parallelism (PP)
- Split model layers sequentially
- GPU 1: layers 0-15
- GPU 2: layers 16-31
- Higher latency (sequential)

### Data Parallelism (DP)
- Replicate entire model
- Each GPU processes different requests
- Highest throughput
- For models that fit on single GPU

**Hybrid**: TP + PP + DP for very large clusters

In [ ]:
def visualize_parallelism():
    """Visualize different parallelism strategies."""
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # 1. Single GPU
    ax1 = axes[0, 0]
    ax1.set_title('Single GPU (Baseline)', fontsize=13, weight='bold')
    ax1.set_xlim(0, 1)
    ax1.set_ylim(0, 1)
    ax1.axis('off')
    
    rect = plt.Rectangle((0.2, 0.3), 0.6, 0.4,
                         facecolor='lightblue', edgecolor='black', linewidth=3)
    ax1.add_patch(rect)
    ax1.text(0.5, 0.6, 'Full Model\n(70B params)', ha='center', va='center',
            fontsize=12, weight='bold')
    ax1.text(0.5, 0.45, 'GPU 1', ha='center', va='center',
            fontsize=11, style='italic')
    
    ax1.text(0.5, 0.15, '✗ Doesn\'t fit!\n(Need 140GB, have 80GB)',
            ha='center', fontsize=10, color='red', weight='bold',
            bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.7))
    
    # 2. Tensor Parallelism
    ax2 = axes[0, 1]
    ax2.set_title('Tensor Parallelism (TP)', fontsize=13, weight='bold')
    ax2.set_xlim(0, 1)
    ax2.set_ylim(0, 1)
    ax2.axis('off')
    
    # 4 GPUs side by side
    tp_colors = ['lightblue', 'lightgreen', 'lightyellow', 'lightcoral']
    for i in range(4):
        x = 0.05 + i * 0.23
        rect = plt.Rectangle((x, 0.3), 0.2, 0.4,
                            facecolor=tp_colors[i], edgecolor='black', linewidth=2)
        ax2.add_patch(rect)
        ax2.text(x + 0.1, 0.6, f'Part\n{i+1}', ha='center', va='center',
                fontsize=10, weight='bold')
        ax2.text(x + 0.1, 0.45, f'GPU\n{i+1}', ha='center', va='center',
                fontsize=9, style='italic')
        ax2.text(x + 0.1, 0.35, f'17.5B', ha='center', va='center',
                fontsize=8)
    
    # Communication arrows
    for i in range(3):
        x1 = 0.05 + i * 0.23 + 0.2
        x2 = 0.05 + (i+1) * 0.23
        ax2.annotate('', xy=(x2, 0.5), xytext=(x1, 0.5),
                    arrowprops=dict(arrowstyle='<->', lw=2, color='red'))
    
    ax2.text(0.5, 0.15, '✓ Low latency\n✓ 70B model fits across 4 GPUs',
            ha='center', fontsize=10, color='green', weight='bold',
            bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))
    
    # 3. Pipeline Parallelism
    ax3 = axes[1, 0]
    ax3.set_title('Pipeline Parallelism (PP)', fontsize=13, weight='bold')
    ax3.set_xlim(0, 1)
    ax3.set_ylim(0, 1)
    ax3.axis('off')
    
    # 4 GPUs stacked
    pp_colors = ['lightblue', 'lightgreen', 'lightyellow', 'lightcoral']
    layer_ranges = ['Layers 0-19', 'Layers 20-39', 'Layers 40-59', 'Layers 60-79']
    
    for i in range(4):
        y = 0.7 - i * 0.18
        rect = plt.Rectangle((0.2, y), 0.6, 0.15,
                            facecolor=pp_colors[i], edgecolor='black', linewidth=2)
        ax3.add_patch(rect)
        ax3.text(0.5, y + 0.105, layer_ranges[i], ha='center', va='center',
                fontsize=10, weight='bold')
        ax3.text(0.5, y + 0.04, f'GPU {i+1}', ha='center', va='center',
                fontsize=9, style='italic')
        
        # Arrow to next GPU
        if i < 3:
            ax3.annotate('', xy=(0.5, y), xytext=(0.5, y - 0.03),
                        arrowprops=dict(arrowstyle='->', lw=3, color='blue'))
    
    ax3.text(0.5, 0.05, '✓ Fits 70B model\n⚠ Higher latency (sequential)',
            ha='center', fontsize=10, weight='bold',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))
    
    # 4. Data Parallelism
    ax4 = axes[1, 1]
    ax4.set_title('Data Parallelism (DP)', fontsize=13, weight='bold')
    ax4.set_xlim(0, 1)
    ax4.set_ylim(0, 1)
    ax4.axis('off')
    
    # 4 full model replicas
    for i in range(4):
        x = 0.05 + i * 0.23
        rect = plt.Rectangle((x, 0.3), 0.2, 0.4,
                            facecolor='lightblue', edgecolor='black', linewidth=2)
        ax4.add_patch(rect)
        ax4.text(x + 0.1, 0.6, 'Full\nModel', ha='center', va='center',
                fontsize=10, weight='bold')
        ax4.text(x + 0.1, 0.45, f'GPU\n{i+1}', ha='center', va='center',
                fontsize=9, style='italic')
        
        # Requests
        ax4.text(x + 0.1, 0.2, f'Req\n{i+1}', ha='center', va='center',
                fontsize=9, bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))
        ax4.annotate('', xy=(x + 0.1, 0.3), xytext=(x + 0.1, 0.24),
                    arrowprops=dict(arrowstyle='->', lw=2, color='green'))
    
    ax4.text(0.5, 0.05, '✓ Highest throughput (4x)\n⚠ Each model must fit on 1 GPU',
            ha='center', fontsize=10, weight='bold',
            bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))
    
    plt.tight_layout()
    plt.show()
    
    print("Distributed Inference Strategies:")
    print("="*70)
    
    print("\n1. TENSOR PARALLELISM (TP):")
    print("   • Split each layer across GPUs")
    print("   • Example: Attention split into 4 parts")
    print("   • Low latency (parallel computation)")
    print("   • High GPU-GPU communication (NVLink required)")
    print("   • Best for: Very large models, low latency")
    
    print("\n2. PIPELINE PARALLELISM (PP):")
    print("   • Split layers sequentially across GPUs")
    print("   • Each GPU processes different layer")
    print("   • Higher latency (sequential)")
    print("   • Lower communication overhead")
    print("   • Best for: Very deep models")
    
    print("\n3. DATA PARALLELISM (DP):")
    print("   • Replicate entire model")
    print("   • Each GPU processes different batch")
    print("   • Highest throughput")
    print("   • No inter-GPU communication")
    print("   • Best for: Models that fit on single GPU")
    
    print("\nHYBRID STRATEGIES:")
    print("   • TP + PP: For very large models (175B+)")
    print("   • TP + DP: High throughput for large models")
    print("   • Example: GPT-3 175B → 8-way TP, 16-way PP")
    
    print("\nPRACTICAL GUIDELINES:")
    print("   • 7B-13B: Single GPU (A100/H100)")
    print("   • 30B-70B: 2-4 way TP")
    print("   • 175B+: TP + PP")
    print("   • High traffic: Add DP replicas")

visualize_parallelism()

## Summary

You've mastered production LLM systems:
- ✅ Model serving architectures
- ✅ Quantization for memory reduction
- ✅ Flash Attention for speed
- ✅ Distributed inference strategies
- ✅ Parallelism techniques (TP, PP, DP)

**Key Insights**:
1. **Quantization**: 4-bit reduces memory 4x with <1% accuracy loss
2. **Flash Attention**: 2-4x faster, enables longer sequences
3. **Tensor Parallelism**: Best for large models, low latency
4. **Production stack**: Load balancer → Queue → vLLM → GPUs
5. **Cost optimization**: Right-size hardware, use quantization, batch efficiently

**Performance Stack**:

| Optimization | Speedup | Memory Saving | Difficulty |
|--------------|---------|---------------|------------|
| Continuous Batching | 2-4x | - | Easy (vLLM) |
| Flash Attention | 2-3x | 8x | Easy (built-in) |
| INT8 Quantization | 1.5x | 2x | Easy (GPTQ) |
| INT4 Quantization | 2x | 4x | Medium (AWQ) |
| Tensor Parallelism | - | 4x+ | Medium |
| Speculative Decoding | 2-3x | - | Hard |

**Cost Examples** (A100 80GB @ $2/hour):
- **LLaMA-7B**: 1 GPU, INT4 → $2/hour, 1K tokens/sec
- **LLaMA-70B**: 4 GPU TP, INT4 → $8/hour, 500 tokens/sec
- **LLaMA-70B**: 8 GPU DP replicas → $16/hour, 8K tokens/sec

**Production Checklist**:
- ✓ Use vLLM or TGI for serving
- ✓ Enable Flash Attention 2
- ✓ Quantize to INT4/INT8
- ✓ Monitor P95/P99 latencies
- ✓ Set request timeouts
- ✓ Implement rate limiting
- ✓ Cache common queries
- ✓ Use autoscaling
- ✓ Deploy multi-region for latency
- ✓ Track cost per 1M tokens

**Popular Tools**:
- **vLLM**: Best throughput, PagedAttention
- **TGI**: Hugging Face, good all-around
- **TensorRT-LLM**: NVIDIA optimized, fastest
- **llama.cpp**: CPU inference, edge devices
- **Ollama**: Easy local deployment

**Next Steps**: LLMOps - monitoring, deployment, A/B testing in production

## Exercises

1. **Implement quantization**: Quantize model to INT8, measure accuracy loss
2. **Benchmark serving**: Compare vLLM vs TGI throughput
3. **Memory analysis**: Calculate memory for different model sizes and quantization
4. **Design serving architecture**: Plan infrastructure for 1M requests/day
5. **Cost optimization**: Find cheapest setup for target throughput
6. **Distributed setup**: Implement tensor parallelism for large model